# hybraut watchdog

The runtime system is monitored and managed by a **Watchdog Finite State Machine (FSM)**. This FSM listens to runtime events and responds accordingly to maintain mission continuity, recover from failures, and safely shut down in critical conditions. It helps enforce lifecycle safety and determinism across automaton operations.

When specific events are published (e.g., mode transitions, errors, mission completion), the FSM interprets these and updates the automaton's state. Each FSM state is also associated with a corresponding **status message**, which is published to inform external systems about the automaton's condition.

![hybraut_watchdog_fsm_png](.github/diagrams/hybraut_watchdog_fsm_flowchart.png)

In [ ]:
""" 
a simple script for testing the hybraut watchdog FSMp
"""

import rclpy
from rclpy.node import Node
from rclpy.qos import QoSProfile, qos_profile_system_default
from rclpy.executors import MultiThreadedExecutor, Executor
import threading
from hybraut_interfaces.msg import AutomatonEvents
import os
# import sys

# # sys.path.append(os.path.dirname(__file__))

# from hybraut_watchdog import HybrautWatchdogFSM
from hybraut_executor.watchdog import HybrautWatchdogFSM

def main():
    rclpy.init()
    executor: Executor = MultiThreadedExecutor(num_threads=os.cpu_count())
    node = Node('mock_node')
    executor.add_node(node)
    
    thread = threading.Thread(target=executor.spin, daemon=True)
    thread.start()
    
    try:
        watchdogFSM: HybrautWatchdogFSM = HybrautWatchdogFSM(node)
        # Test 1. transition to TRANSITIONING STATE
        watchdogFSM.trigger_transition(AutomatonEvents(
            type=AutomatonEvents.TRANSITION_GUARD_ENABLED
        ))
        #Test 2: Transition back to active state
        watchdogFSM.trigger_transition(AutomatonEvents(
            type=AutomatonEvents.TRANSITION_COMPLETE
        ))
        # Test 3: Transition to error state
        watchdogFSM.trigger_transition(AutomatonEvents(
            type=AutomatonEvents.RECOVERABLE_ERROR
        ))
        
    except KeyboardInterrupt:
        print ("Shutting down gracefully...")
    finally:
        executor.shutdown()
        thread.join()
        node.destroy_node()
        
    rclpy.shutdown()

if __name__ == '__main__':
    main()

ImportError: cannot import name 'StatusBus' from 'comm' (/home/ryan/.local/lib/python3.10/site-packages/comm/__init__.py)